In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import InMemoryVectorStore

/tmp/ipykernel_180133/535478846.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
loader = PyPDFLoader("../data/medical_report.pdf")
doc = loader.load()

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitter_data = splitter.split_documents(doc)

In [5]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [6]:
vector_store =  InMemoryVectorStore.from_documents(
    documents=splitter_data,
    embedding=embeddings
)

### Agent =  tools, llm, prompt

In [16]:
from langchain.tools import tool

@tool
def retriever_tool(query:str):
    """
    This toole can help you to retrive the relevant data of the PDF 
    documents and these document have details about medical reports
    """
    docs =  vector_store.similarity_search(query=query, k=4)
    context = ""

    for doc in docs:
        context += doc.page_content+"\n\n"

    return context

In [17]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")


In [18]:
System_Prompt = """
  You are a helpful assistant that answers questions using retrieved context.
  Always use the 'retriever_tool' tool fo the questions requiring external knowledge.
"""

In [21]:
from langchain.agents import create_agent
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)


In [22]:
query = "What is the name of patient, and what is the name of Doctors"
response = agent.invoke({"messages":[{"role":"user", "content": query}]})

In [26]:
result = response["messages"][-1].content
print(result)

[{'type': 'text', 'text': 'Based on the medical report:\n\n* **Patient Name:** Ms. NIKITA CHUDHARY\n* **Doctors / Referring Physicians mentioned:**\n  * **Ref By:** Dr. Nitin Nahar\n  * **Other Doctors/Pathologists listed in the report:** \n    * Dr. Beena Chandrasekhar\n    * Dr. Nagarjun Sai Jaine\n    * Dr. Sunanda', 'extras': {'signature': 'EmAKXgFpFH0TGjJYtklvulXb0kGByZFkvIH/kXTpIlslGWiQ1MhNQP5T83mC7T4NVSkxasLYF3DmuWhXYbEul32JU2P5uZZadLvDnS47NZzszIBew2gGOCGaWUk0rw+zLPM='}}]
